# Catálogo común de variedades y equivalencias

Único catálogo analítico compartido por auditoría de curvas, proyecciones y homologación de la base analítica. Se genera desde los CSV originales de SQL, sin modificarlos, en `01_maestros_y_geografia/Catalogo_analitico`. Conserva todos los IDs y agrupa únicamente combinaciones exactas de producto/color/nombre normalizados. Múltiples IDs de una combinación no autorizan a escoger uno como código oficial.

`equivalencias.json` documenta las traducciones explícitas. Una equivalencia nueva requiere evidencia; no se usa semejanza difusa. La versión actual del maestro no acredita su disponibilidad histórica. Si cambia el maestro o una regla, ejecutar este notebook y reconstruir la cadena. La limpieza de producción/estimados conserva colores reportados: este catálogo no debe sobrescribirlos sin evidencia.


La clave representa una categoría comercial, no una certificación de identidad genética. Se exponen códigos PickObtentor diferentes dentro de la misma combinación; no se traducen a nombres sin un catálogo verificado. Estas discrepancias requieren revisión agronómica y no se borran al agrupar comercialmente.

In [1]:
from pathlib import Path
import json
import hashlib
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'Datos_analiticos_proyecto').exists())
MAESTROS=ROOT/'Notebooks/Extracción SQL/Datos/01_maestros_y_geografia'
CATALOGO=MAESTROS/'Catalogo_analitico'; CATALOGO.mkdir(exist_ok=True)
def normalizar(s):
    return s.astype('string').str.strip().str.upper().str.normalize('NFKD').str.encode('ascii',errors='ignore').str.decode('ascii').str.replace(r'\s+',' ',regex=True)
productos=pd.read_csv(MAESTROS/'TablasBasicas_Producto.csv',usecols=['IdProducto','NomProducto'])
colores=pd.read_csv(MAESTROS/'TablasBasicas_Color.csv',usecols=['IdColor','NomColor'])
variedades=pd.read_csv(MAESTROS/'TablasBasicas_Variedad.csv',usecols=['IdVariedad','IdProducto','IdColor','NomVariedad','PickObtentor'])
maestro=variedades.merge(productos,on='IdProducto',validate='many_to_one').merge(colores,on='IdColor',validate='many_to_one')
assert len(maestro)==len(variedades) and maestro.IdVariedad.is_unique
for campo,nombre in [('_prod','NomProducto'),('_color','NomColor'),('_var','NomVariedad')]:maestro[campo]=normalizar(maestro[nombre])
ALIAS_PRODUCTOS = {'CLAVEL':'CARNATION', 'MINICLAVEL':'MINICARNATION', 'VERONICAS':'VERONICA'}
ALIAS_COLORES = {'ROJO':'RED','BLANCO':'WHITE','AMARILLO':'YELLOW','VERDE':'GREEN',
    'NARANJA':'ORANGE','CREMA':'CREAM','MORADO':'PURPLE','CEREZA':'CHERRY',
    'BICOLOR ROJO':'BICOLOR RED','BICOLOR AMARILLO':'BICOLOR YELLOW',
    'BICOLOR MORADO':'BICOLOR PURPLE','BICOLOR NARANJA':'BICOLOR ORANGE',
    'BICOLOR BLANCO':'BICOLOR WHITE','BICOLOR CEREZA':'BICOLOR CHERRY'}

maestro.to_parquet(CATALOGO/'maestro_variedades_integrado.parquet',index=False)
productos.to_parquet(CATALOGO/'productos.parquet',index=False)
colores.to_parquet(CATALOGO/'colores.parquet',index=False)
(CATALOGO/'equivalencias.json').write_text(json.dumps({'productos':ALIAS_PRODUCTOS,'colores':ALIAS_COLORES},ensure_ascii=False,indent=2),encoding='utf-8')
identidades=maestro.groupby(['_prod','_color','_var'],dropna=False).agg(codigos=('IdVariedad',lambda s:';'.join(str(int(x)) for x in sorted(s))),numero_ids=('IdVariedad','size')).reset_index()
revision_obtentor=maestro.groupby(['_prod','_color','_var'],dropna=False).agg(codigos_obtentor=('PickObtentor',lambda s:';'.join(str(x) for x in sorted(s.dropna().unique()))),numero_codigos_obtentor=('PickObtentor','nunique')).reset_index()
identidades=identidades.merge(revision_obtentor,on=['_prod','_color','_var'],validate='one_to_one')
identidades['revisar_obtentor']=identidades.numero_codigos_obtentor.gt(1)
identidades['clave_curva']='COMERCIAL|'+identidades._prod+'|'+identidades._color+'|'+identidades._var
identidades.to_parquet(CATALOGO/'identidades_comerciales.parquet',index=False)
identidades.to_excel(CATALOGO/'revision_identidades_comerciales.xlsx',index=False)
huellas={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in [MAESTROS/'TablasBasicas_Producto.csv',MAESTROS/'TablasBasicas_Color.csv',MAESTROS/'TablasBasicas_Variedad.csv']}
(CATALOGO/'fuentes_sha256.json').write_text(json.dumps(huellas,indent=2),encoding='utf-8')
print('IDs de variedad conservados:',len(maestro),'Identidades comerciales:',len(identidades))
display(identidades.loc[identidades.numero_ids.gt(1)].sort_values('numero_ids',ascending=False).head(25))
print('No se selecciona un código SQL representante. Los colores distintos siguen siendo identidades distintas.')


IDs de variedad conservados: 2974 Identidades comerciales: 2915


,_prod,_color,_var,codigos,numero_ids,codigos_obtentor,numero_codigos_obtentor,revisar_obtentor,clave_curva
1288,CARNATION,YELLOW,ILIAS,550;553;559;565,4,0,1,False,COMERCIAL|CARNATION|YELLOW|ILIAS
1771,MINICARNATION,BICOLOR PINK,VALENTINE,5301;5307;6407,3,263;269;271,3,True,COMERCIAL|MINICARNATION|BICOLOR PINK|VALENTINE
2791,SOLOMIO,LIGHT PINK,CAS,1123;1125;1126,3,0;272,2,True,COMERCIAL|SOLOMIO|LIGHT PINK|CAS
1304,CARNATION,YELLOW,XXILIAS,560;564;566,3,0,1,False,COMERCIAL|CARNATION|YELLOW|XXILIAS
697,CARNATION,LIGHT BROWN,HONEY,5300;7859;8507,3,263;269,2,True,COMERCIAL|CARNATION|LIGHT BROWN|HONEY
209,CARNATION,BICOLOR ORANGE,MILONGA,7856;7858,2,263;277,2,True,COMERCIAL|CARNATION|BICOLOR ORANGE|MILONGA
222,CARNATION,BICOLOR PINK,BLIS,5280;5298,2,0,1,False,COMERCIAL|CARNATION|BICOLOR PINK|BLIS
256,CARNATION,BICOLOR PINK,LEGE SPORT,84;5271,2,0;263,2,True,COMERCIAL|CARNATION|BICOLOR PINK|LEGE SPORT
248,CARNATION,BICOLOR PINK,LADY DAIKI,81;83,2,0,1,False,COMERCIAL|CARNATION|BICOLOR PINK|LADY DAIKI
285,CARNATION,BICOLOR PINK,YUCARI CHERRY,6427;7818,2,263;277,2,True,COMERCIAL|CARNATION|BICOLOR PINK|YUCARI CHERRY


No se selecciona un código SQL representante. Los colores distintos siguen siendo identidades distintas.
